# Telco churn — EDA and results walkthrough

This notebook explores the IBM Telco dataset, motivates the modelling choices made in `src/churnintel`, and
reads back the results produced by `make train`. Everything heavy lives in the package; the notebook only
*uses* it.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from churnintel.data import load_dataset, RAW_FEATURES
from churnintel.viz.style import apply_style, BLUE, ORANGE, AQUA, MUTED
apply_style()
pd.set_option("display.precision", 3)

X, y = load_dataset(ROOT / "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = X.assign(Churn=y)
print(df.shape, f"churn rate = {y.mean():.1%}")

(7043, 20) churn rate = 26.5%


## 1. Who churns? Churn rate by key segments

In [2]:
def rate(col):
    g = df.groupby(col)["Churn"].agg(customers="size", churn_rate="mean")
    return g.sort_values("churn_rate", ascending=False)

for col in ["Contract", "InternetService", "PaymentMethod"]:
    display(rate(col))

,customers,churn_rate
Contract,,
Month-to-month,3875,0.427
One year,1473,0.113
Two year,1695,0.028


,customers,churn_rate
InternetService,,
Fiber optic,3096,0.419
DSL,2421,0.190
No,1526,0.074


,customers,churn_rate
PaymentMethod,,
Electronic check,2365,0.453
Mailed check,1612,0.191
Bank transfer (automatic),1544,0.167
Credit card (automatic),1522,0.152


In [3]:
df["tenure_band"] = pd.cut(df["tenure"], [-1, 6, 12, 24, 48, 72], labels=["0-6", "7-12", "13-24", "25-48", "49-72"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
r = df.groupby("tenure_band", observed=True)["Churn"].mean()
axes[0].bar(r.index.astype(str), r.values, color=BLUE, edgecolor="#fcfcfb", linewidth=2)
axes[0].set_title("Churn rate by tenure band (months)"); axes[0].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
axes[0].grid(axis="x", visible=False)
for label, color in [(0, BLUE), (1, ORANGE)]:
    axes[1].hist(df.loc[df.Churn == label, "MonthlyCharges"], bins=30, alpha=0.6, color=color, density=True,
                 label="stayed" if label == 0 else "churned")
axes[1].set_title("Monthly charges by outcome"); axes[1].legend(); axes[1].set_xlabel("$ per month")
plt.show()

**Takeaways:** month-to-month contracts, fiber-optic internet, electronic-check payment and the first
6–12 months are the high-risk segments. Churners skew towards higher monthly bills.

## 2. Why some engineered features were removed: MonthlyCharges is (almost) a function of services

In [4]:
from sklearn.linear_model import LinearRegression
services = pd.get_dummies(df[["PhoneService", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
                              "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]], drop_first=True).astype(float)
r2 = LinearRegression().fit(services, df["MonthlyCharges"]).score(services, df["MonthlyCharges"])
print(f"R^2 of MonthlyCharges ~ subscribed services: {r2:.4f}")

R^2 of MonthlyCharges ~ subscribed services: 0.9988


With R² ≈ 1, features such as `Fiber_x_Charge`, `ServiceDensity` or `NumServices` (v3) are near-linear
combinations of other columns. A logistic model can then trade huge positive weights on services against a
huge negative weight on price, which makes SHAP reason codes contradictory. v4 drops them (ΔAUC < 0.001) and
picks the logistic-regression penalty with a paired one-standard-error rule.

## 3. Results from `make train`

In [5]:
m = json.loads((ROOT / "reports/metrics.json").read_text())
t = m["test"]["champion"]; ci = t["bootstrap_ci"]
print(f"Champion: {m['selection']['champion_label']} ({m['calibration']['method']} calibration)")
print(f"Test ROC-AUC {t['roc_auc']:.4f}  (95% CI {ci['roc_auc']['ci_lower']:.4f}-{ci['roc_auc']['ci_upper']:.4f})")
print(f"Test PR-AUC {t['pr_auc']:.4f} | Brier {t['brier']:.4f} | top-10% lift {t['lift']['top_10pct']['lift']:.2f}x")
pd.read_csv(ROOT / "reports/model_comparison.csv", index_col=0)[
    ["model", "roc_auc_mean", "roc_auc_std", "brier_mean", "test_roc_auc", "test_pr_auc"]
].sort_values("roc_auc_mean", ascending=False)

Champion: Logistic Regression (none calibration)
Test ROC-AUC 0.8497  (95% CI 0.8284-0.8690)
Test PR-AUC 0.6728 | Brier 0.1335 | top-10% lift 2.91x


,model,roc_auc_mean,roc_auc_std,brier_mean,test_roc_auc,test_pr_auc
stacking,Stacking Ensemble,0.851,0.012,0.134,0.849,0.671
soft_voting,Soft-Voting Ensemble,0.851,0.012,0.133,0.849,0.671
logreg,Logistic Regression,0.851,0.012,0.133,0.850,0.673
catboost,CatBoost,0.850,0.012,0.133,0.848,0.668
xgboost,XGBoost,0.850,0.012,0.134,0.847,0.665
lightgbm,LightGBM,0.849,0.013,0.134,0.848,0.666
random_forest,Random Forest,0.847,0.013,0.135,0.847,0.660


In [6]:
pd.read_csv(ROOT / "reports/policy_comparison.csv", index_col=0)[
    ["customers_targeted", "churner_capture_rate", "campaign_cost", "net_profit", "net_profit_per_1k_customers", "roi"]
]

,customers_targeted,churner_capture_rate,campaign_cost,net_profit,net_profit_per_1k_customers,roi
policy,,,,,,
No campaign,0,0.000,0.000,0.000,0.000,NaN
Blanket campaign (contact everyone),1409,1.000,105179.000,-5861.555,-4160.081,-0.056
ML threshold policy (p >= 0.33),497,0.757,31208.000,42604.742,30237.574,1.365
Rule-based heuristic (same volume),497,0.706,31949.000,34233.495,24296.306,1.072
"Random targeting (same volume, expected)",497,0.353,37100.045,-2067.560,-1467.396,-0.056
ML next-best-action (EV-optimal),628,0.789,33134.800,43787.106,31076.725,1.321
"Oracle (perfect foresight, reference)",374,1.000,17204.000,82113.445,58277.818,4.773


## 4. Scoring customers with reason codes (same code path as the API)

In [7]:
from churnintel.inference import ChurnPredictor
predictor = ChurnPredictor.load(ROOT / "artifacts/model_bundle.joblib")
test_ids = pd.read_csv(ROOT / "reports/test_predictions.csv")["customerID"]
sample = X.loc[test_ids].sample(5, random_state=3)
scored = predictor.score(sample, explain=True)
scored["top_reasons"] = [" | ".join(r["description"] for r in e["risk_drivers"]) for e in scored.pop("explanation")]
scored[["churn_probability", "risk_tier", "clv_estimate", "recommended_action", "expected_value", "top_reasons"]]

,churn_probability,risk_tier,clv_estimate,recommended_action,expected_value,top_reasons
customerID,,,,,,
5360-LJCNJ,0.117,low,1486.94,email_offer,1.04,High monthly bill ($105) | Fiber optic interne...
7729-XBTWX,0.011,low,256.83,no_action,0.00,No internet add-ons (no internet service) | Pa...
2845-HSJCY,0.634,high,1036.21,retention_call,169.66,Fiber optic internet | High monthly bill ($87)...
8338-QIUNR,0.015,low,957.60,no_action,0.00,Paperless billing | Multiple phone lines | Mon...
2263-SFSQZ,0.020,low,344.98,no_action,0.00,No internet add-ons (no internet service) | Pa...


## 5. Dashboard

![dashboard](../reports/figures/dashboard.png)